In [ ]:
dbutils.widgets.text("catalog", "", "Catalog Name")
dbutils.widgets.text("schema", "", "Schema Name")
dbutils.widgets.text("volume", "", "Volume Name")

Box(children=(Label(value='Catalog Name'), Text(value='')))

Box(children=(Label(value='Schema Name'), Text(value='')))

Box(children=(Label(value='Volume Name'), Text(value='{{var.volume}}')))

In [5]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

""


In [6]:
# Generate customer data using dbldatagen
import dbldatagen as dg

# roughly 300 customers with unique customer_id
cust_gen = (
    dg.DataGenerator(spark, name="customers", rows=300)
      .withColumn("customer_id", "integer", minValue=1, maxValue=300, uniqueValues=300)
      .withColumn("first_name", "string", values=["Alice", "Bob", "Carol", "Dave", "Eve"], random=True)
      .withColumn("last_name", "string", values=["Smith", "Jones", "Taylor", "Brown", "Wilson"], random=True)
)
df_customers = cust_gen.build()
df_customers.show(5)

+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|          1|       Bob|    Smith|
|          2|     Carol|    Smith|
|          3|       Eve|   Wilson|
|          4|       Bob|    Smith|
|          5|       Bob|    Jones|
+-----------+----------+---------+
only showing top 5 rows


In [7]:
import time

# Generate order data using dbldatagen
# roughly 1000 orders; only use customer_ids generated above
customer_ids = [row.customer_id for row in df_customers.select("customer_id").collect()]

order_gen = (
    dg.DataGenerator(spark, name="orders", rows=1000)
      .withColumn("order_id", "integer", minValue=1, maxValue=1000, uniqueValues=1000)
      .withColumn("customer_id", "integer", values=customer_ids, random=True)
      .withColumn("order_date", "timestamp", minValue=time.time() - 365*24*3600, maxValue=time.time())  # last year
      .withColumn("order_status", "string", values=["NEW", "SHIPPED", "DELIVERED", "CANCELLED"], random=True)
)
df_orders = order_gen.build()
df_orders.show(5)

+--------+-----------+-------------------+------------+
|order_id|customer_id|         order_date|order_status|
+--------+-----------+-------------------+------------+
|       1|        204|2024-12-31 23:00:00|     SHIPPED|
|       2|         91|2025-01-01 23:00:00|   CANCELLED|
|       3|        117|2025-01-02 23:00:00|     SHIPPED|
|       4|        130|2025-01-03 23:00:00|     SHIPPED|
|       5|          6|2025-01-04 23:00:00|         NEW|
+--------+-----------+-------------------+------------+
only showing top 5 rows


In [8]:
import random
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType, TimestampType
from datetime import timedelta
from decimal import Decimal
# Generate payments data manually with correlated status sequences

payment_rows = []
payment_id = 1000

for oid in df_orders.collect():
    # 1 to 3 events per order
    n = random.randint(1, 3)
    if n == 1:
        statuses = [random.choice(["FAILED", "SUCCESS"])]
    else:
        statuses = ["PENDING"]
        statuses.append(random.choice(["FAILED", "SUCCESS"]))
    payment_method = random.choice(["CREDIT_CARD", "PAYPAL", "WIRE"])
    for status in statuses:
        # order_date is a datetime, add seconds with timedelta
        timestamp = oid.order_date + timedelta(seconds=random.randint(1, 10) * 60)
        payment_rows.append(
            (
                payment_id,
                oid.order_id,
                payment_method,
                status,
                Decimal(str(round(random.uniform(1.0, 500.0), 2))),
                timestamp,
            )
        )
        timestamp += timedelta(seconds=random.randint(10, 20) * 60)
        payment_id += 1

schemaTypes = StructType([
    StructField("payment_id", IntegerType(), False),
    StructField("order_id", IntegerType(), False),
    StructField("payment_method", StringType(), False),
    StructField("payment_status", StringType(), False),
    StructField("payment_amount", DecimalType(10, 2), False),
    StructField("created", TimestampType(), False),
])

df_payments = spark.createDataFrame(payment_rows, schemaTypes)
df_payments.show(5)

+----------+--------+--------------+--------------+--------------+-------------------+
|payment_id|order_id|payment_method|payment_status|payment_amount|            created|
+----------+--------+--------------+--------------+--------------+-------------------+
|      1000|       1|   CREDIT_CARD|       PENDING|        350.93|2024-12-31 23:05:00|
|      1001|       1|   CREDIT_CARD|        FAILED|        408.82|2024-12-31 23:03:00|
|      1002|       2|          WIRE|        FAILED|        264.90|2025-01-01 23:01:00|
|      1003|       3|          WIRE|        FAILED|        224.94|2025-01-02 23:04:00|
|      1004|       4|          WIRE|       PENDING|        314.13|2025-01-03 23:01:00|
+----------+--------+--------------+--------------+--------------+-------------------+
only showing top 5 rows


In [9]:
# verify every order has a matching customer_id
unlinked = (
    df_orders.join(df_customers, on="customer_id", how="left_anti")
    .select("order_id", "customer_id")
)
if unlinked.count() == 0:
    print("All orders have valid customer_id links.")
else:
    print(f"Found {unlinked.count()} orders without matching customer_id:")
    display(unlinked.limit(20))
    raise ValueError("Data integrity check failed: some orders have invalid customer_id.")

All orders have valid customer_id links.


In [10]:
# verify every payment has a matching order_id
unlinked = (
    df_payments.join(df_orders, on="order_id", how="left_anti")
    .select("order_id", "payment_id")
)
if unlinked.count() == 0:
    print("All payments have valid order_id links.")
else:
    print(f"Found {unlinked.count()} payments without matching order_id:")
    display(unlinked.limit(20))
    raise ValueError("Data integrity check failed: some payments have invalid order_id.")

All payments have valid order_id links.


In [11]:
from governer.utils import save_frame_csv, validate_schema
from uuid import uuid4

from governer.schemas.raw.customers import customers as raw_customers
from governer.schemas.raw.orders import orders as raw_orders
from governer.schemas.raw.payments import payments as raw_payments

orders_output_path = f"/Volumes/{catalog}/{schema}/{volume}/orders/clean_{uuid4()}"
customers_output_path = f"/Volumes/{catalog}/{schema}/{volume}/customers/clean_{uuid4()}" 
payments_output_path = f"/Volumes/{catalog}/{schema}/{volume}/payments/clean_{uuid4()}"


df_orders = validate_schema(df_orders, raw_orders)
df_customers = validate_schema(df_customers, raw_customers)
df_payments = validate_schema(df_payments, raw_payments)

save_frame_csv(df_orders, orders_output_path, dbutils)
save_frame_csv(df_customers, customers_output_path, dbutils)
save_frame_csv(df_payments, payments_output_path, dbutils)



'/Volumes/governer/uharabtsou/source/payments/clean_bc4f87a4-8f2a-4027-a76d-32535e5c0dcc.csv'